# Running Statistical Tests in Python — t-test & Chi-Square

**Module 5: Inferential Statistics**

**Learning Objectives:**
- Following a step-by-step template, run a two-sample t-test or a chi-square test (as appropriate to the data type) in Python via scipy.stats, correctly stating the null hypothesis, computing the p-value, and reading off the 95% confidence interval

**Skills:** t-test, chi-square test, contingency table, degrees of freedom, categorical variable


> **You are not presenting this notebook to students.** Use it as your own answer key while you fill in a student notebook live, in front of the class.


## Scenario

Yesterday you stated a null and alternative hypothesis for one real question: Do Gentoo penguins actually weigh more than Adelie penguins? Today you stop guessing and find out - by running the real test, in Python, on the real data.


## Documentation

- Run the t-test: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_ind.html

- Run the Chi-Square Test: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chi2_contingency.html


## Load the Real Data

*Context: Real Palmer Penguins data has a few genuinely missing measurements - dropping rows missing the one column you're about to test keeps every later step honest.*

**Instructor notes:**

> [SCRIPT] Name the new tool before typing a line of it. `scipy.stats` is the real, standard Python library data analysts reach for to run hypothesis tests - everything today's demo uses lives inside it.
> 
> [ACTIVITY] Type this along with the instructor. Confirm your own `df.head()` output shows real rows before moving on - species, island, and body_mass_g should all be visible.
> 
> [SCRIPT] Name why the drop matters now, not later. A few penguins are missing a body-mass measurement. Testing with those rows still in would silently break the comparison - drop them once, up front, for the column you're about to use.
> 
> [MISCONCEPTIONS] Someone will ask why not just drop every row with any missing value anywhere. That would throw away real, usable rows whose only gap is in a column you're not even testing today.

*Input:*


In [ ]:
import pandas as pd
from scipy import stats

df = pd.read_csv("palmer_penguins.csv")
df = df.dropna(subset=["body_mass_g"])
df.head()


## The Shape Check, in Code

**Instructor notes:**

> [ACTIVITY] Run this, then say out loud what you see: are the three species' spreads (the `std` column) roughly similar, or is one way more spread out than the others?
> 
> [SCRIPT] Point at `mean` and `std` together, not just one. Adelie and Gentoo should look different enough in `mean` to be interesting, without either group's `std` looking wildly larger than the other's.
> 
> [MISCONCEPTIONS] People expect "roughly similar shape" to mean the three `std` values match almost exactly. It means no group's spread is wildly larger than the others - real groups never match exactly, and demanding that they do would fail almost any real dataset.

*Input:*


In [ ]:
df.groupby("species")["body_mass_g"].describe()


### 🤔 Check for Understanding: Restate the Null, Right Before Testing

*Mode: Turn and Talk*

With your partner: State the null hypothesis you'll test next, in one sentence.

**Instructor notes:**

> [ACTIVITY] Pairs say their sentence out loud to each other before the instructor advances. In person: circulate and listen for "no real difference" language. Online: post in chat. 2 minutes.
> 
> [SCRIPT] Target answer: "There is no real difference in average body mass between Adelie and Gentoo penguins." Confirm the room has this right before the real test result lands - it is much easier to rationalize a result after seeing it than before.


## Run the t-test

*Context: `.confidence_interval()` is a real method on the result scipy.stats.ttest_ind returns - it hands back the same 95% CI concept from last class, computed for real from this exact comparison.*

**Instructor notes:**

> [ACTIVITY] Run it, then read your own three numbers out loud to your partner before the instructor reveals the expected output.
> 
> [SCRIPT] Walk the real output line by line. The p-value here is far smaller than 0.05 - small enough that "this is just random sampling noise" is not a credible explanation anymore. The 95% CI lands entirely on one side of zero, meaning the real direction of the difference is clear, not just its existence.
> 
> [MISCONCEPTIONS] A tiny p-value gets read as "an even more true" result than a p-value of 0.04. Both simply clear the same threshold - how far past it a p-value sits doesn't scale to "more real."

*Input:*


In [ ]:
adelie = df[df["species"] == "Adelie"]["body_mass_g"]
gentoo = df[df["species"] == "Gentoo"]["body_mass_g"]

result = stats.ttest_ind(adelie, gentoo)
print("t-statistic:", result.statistic)
print("p-value:", result.pvalue)
print("95% CI:", result.confidence_interval(confidence_level=0.95))


## Compute Cohen's d

*Context: Cohen's d here comes out well above 0.8 - by the standard convention (0.2 small, 0.5 medium, 0.8 large), this isn't just statistically real, it's a large real-world difference too.*

**Instructor notes:**

> [ACTIVITY] Run it, then say the number out loud: is it closer to 0.2, 0.5, or 0.8 and above?
> 
> [SCRIPT] Land the real number. Cohen's d here comes out well above 0.8 - large by the standard convention. This is the rare case where statistical significance and practical significance both point the same direction, clearly.

*Input:*


In [ ]:
import numpy as np

n1, n2 = len(adelie), len(gentoo)
pooled_std = np.sqrt(
    ((n1 - 1) * adelie.std()**2 + (n2 - 1) * gentoo.std()**2)
    / (n1 + n2 - 2)
)
cohens_d = (gentoo.mean() - adelie.mean()) / pooled_std
print("Cohen's d:", cohens_d)


### 🤔 Check for Understanding: Predict It First: Species and Island

*Mode: Poll*

With your partner: Predict whether species and island are related, before running any code.

**Instructor notes:**

> [ACTIVITY] Pairs commit to a real prediction and one reason for it, out loud, before the instructor runs the code. In person: quick show of hands, related or independent. Online: type your guess in chat. 2 minutes.
> 
> [SCRIPT] Do not confirm or deny yet. The whole point is committing to a prediction before the real crosstab makes the answer obvious - that is what makes the reveal actually land.


## Run the Chi-Square Test

*Context: `expected` is the assumption check for this test: every expected cell count needs to be at least 5 for the chi-square approximation to be defensible - the same 'check before you trust it' habit as the t-test's shape check, just a different rule for a different data type.*

**Instructor notes:**

> [ACTIVITY] Look at the printed `table` before reading any test output. Does it match what you predicted a moment ago?
> 
> [SCRIPT] Walk the real crosstab first, before the test numbers. Some species show up on only one or two islands, not all three - that pattern alone is a strong visual hint before a single p-value is computed.
> 
> [SCRIPT] Then walk the real output. The p-value here is extremely small - species and island are not independent in this data. Name the expected-counts check explicitly: every value in `expected` needs to be at least 5 for this result to be trustworthy.
> 
> [SCRIPT] Name `dof` in plain terms before moving on. Degrees of freedom here is (rows - 1) times (columns - 1) - a number the test needs internally to know which chi-square distribution to compare the statistic against. You won't calculate it by hand again; just recognize it as part of the standard output, not a separate concept to master today.
> 
> [MISCONCEPTIONS] People assume a chi-square test can use raw counts directly. It needs a contingency table first - counts cross-tabbed between the two categorical variables, not a flat list of category labels.

*Input:*


In [ ]:
table = pd.crosstab(df["species"], df["island"])
print(table)

chi2, p, dof, expected = stats.chi2_contingency(table)
print("chi2:", chi2)
print("p-value:", p)
print("degrees of freedom:", dof)
print("expected counts:")
print(expected)


## Compute Cramér's V

*Context: A Cramér's V well above 0.5 is considered a strong association - species and island aren't just 'probably related,' they're strongly related.*

**Instructor notes:**

> [ACTIVITY] Run it, then compare the number to the 0-to-1 scale from the last slide - is this closer to "barely related" or "strongly related"?
> 
> [SCRIPT] Land the real number. Cramér's V here lands well above 0.5 - a strong association, not a marginal one. Same lesson as the t-test side: the p-value said "probably real," the effect size says "and it's a big one."

*Input:*


In [ ]:
n = table.sum().sum()
cramers_v = (chi2 / (n * (min(table.shape) - 1))) ** 0.5
print("Cramer's V:", cramers_v)
